# GeoLife CP2 — Home / Office / POI Baseline

This notebook starts **after the frozen CP1 stay-point pipeline**.

Goals:

1. materialize all CP1 stays with user/file lineage;
2. reconcile the stay total with the CP1 full-release result;
3. audit user-level history sufficiency;
4. inspect recurring-location representation;
5. resolve the timezone/geography policy **before** applying Home/Office time-of-day heuristics;
6. only then define an interpretable Home / Office / Other baseline.

> Home/Office are sensitive derived locations. Do not commit precise user-level inferred-location outputs.

Design contract: `docs/design/03_home_office_baseline_contract.md`.

In [ ]:
from pathlib import Path
from time import perf_counter
from zoneinfo import ZoneInfo
from IPython.display import display
import os
import pickle
import subprocess
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.cluster import AgglomerativeClustering, DBSCAN

REPO_URL = "https://github.com/tanh1c/geolife.git"
REPO_BRANCH = os.environ.get("GEOLIFE_REPO_BRANCH", "cp2-home-office-baseline")
REPO_DIR = Path(os.environ.get("GEOLIFE_REPO_DIR", "/tmp/geolife"))
VOLUME_ROOT = Path("/mnt/geolife-data")
CACHE_DIR = VOLUME_ROOT / "cache" / "cp2_home_office"
CACHE_DIR.mkdir(parents=True, exist_ok=True)

def resolve_data_root():
    env_root = os.environ.get("GEOLIFE_DATA_ROOT")
    candidates = ([Path(env_root)] if env_root else []) + [
        VOLUME_ROOT / "extracted" / "Geolife Trajectories 1.3" / "Data",
        VOLUME_ROOT / "Data",
    ]
    for candidate in candidates:
        if candidate.is_dir() and any(candidate.glob("*/Trajectory/*.plt")):
            return candidate
    for candidate in sorted(VOLUME_ROOT.glob("**/Data")):
        if candidate.is_dir() and any(candidate.glob("*/Trajectory/*.plt")):
            return candidate
    raise FileNotFoundError("GeoLife Data folder not found")

def ensure_repo():
    if (REPO_DIR / ".git").exists():
        subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "origin"], check=True)
        subprocess.run(["git", "-C", str(REPO_DIR), "checkout", REPO_BRANCH], check=True)
        subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only", "origin", REPO_BRANCH], check=True)
    else:
        subprocess.run(["git", "clone", "--branch", REPO_BRANCH, REPO_URL, str(REPO_DIR)], check=True)

DATA_ROOT = resolve_data_root()
ensure_repo()
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
if str(REPO_DIR / "src") not in sys.path:
    sys.path.insert(0, str(REPO_DIR / "src"))

from notebooks.eda_core import read_plt
from geolife.geo.distance import haversine_m
from geolife.staypoints import clean_trajectory, detect_staypoints

BASELINE = {
    "same_second_radius_m": 10.0,
    "max_gap_s": 300.0,
    "hard_speed_guard_kmh": 1200.0,
    "distance_threshold_m": 200.0,
    "min_dwell_s": 1200.0,
}

files = sorted(DATA_ROOT.glob("*/Trajectory/*.plt"))
print("Repo branch:", REPO_BRANCH)
print("Data root:", DATA_ROOT)
print("Trajectory files:", f"{len(files):,}")
print("Cache dir:", CACHE_DIR)
print("Frozen CP1 baseline:", BASELINE)

## 1. Materialize the frozen full-release stay table

CP1 validated **5,821 stays** across all 18,670 trajectory files, but the old cache stored only per-file summaries.

CP2 needs actual stay events grouped by user, so this section materializes them once and caches them.

The loop supports a partial checkpoint. If the kernel stops, rerun the cell and it resumes from already processed files. The final private cache uses pandas pickle (`.pkl`) so notebook execution does not depend on an optional Parquet engine such as `pyarrow`.

In [ ]:
STAYS_CACHE = CACHE_DIR / "stays_baseline_v1.pkl"
PARTIAL_CACHE = CACHE_DIR / "stays_baseline_v1.partial.pkl"
EXPECTED_CP1_STAYS = 5821

def user_id_from_path(path):
    return path.parent.parent.name

def process_file(path):
    raw = read_plt(path)[["timestamp", "latitude", "longitude"]]
    cleaned = clean_trajectory(
        raw,
        same_second_radius_m=BASELINE["same_second_radius_m"],
        max_gap_s=BASELINE["max_gap_s"],
        hard_speed_guard_kmh=BASELINE["hard_speed_guard_kmh"],
    )
    stays = detect_staypoints(
        cleaned,
        distance_threshold_m=BASELINE["distance_threshold_m"],
        min_dwell_s=BASELINE["min_dwell_s"],
    )
    if stays.empty:
        return []
    user_id = user_id_from_path(path)
    out = []
    for row in stays.itertuples(index=False):
        out.append({
            "user_id": user_id,
            "source_file": str(path),
            "sequence_id": int(row.sequence_id),
            "arrival_time_utc": row.arrival_time,
            "departure_time_utc": row.departure_time,
            "duration_s": float(row.duration_s),
            "latitude": float(row.latitude),
            "longitude": float(row.longitude),
            "n_points": int(row.n_points),
        })
    return out

if STAYS_CACHE.exists():
    stays = pd.read_pickle(STAYS_CACHE)
    print("Loaded:", STAYS_CACHE)
else:
    if PARTIAL_CACHE.exists():
        with PARTIAL_CACHE.open("rb") as f:
            partial = pickle.load(f)
        processed = set(partial["processed_files"])
        rows = list(partial["rows"])
        print("Resuming partial:", f"{len(processed):,}/{len(files):,} files")
    else:
        processed = set()
        rows = []

    t0 = perf_counter()
    completed_this_run = 0

    for path in files:
        key = str(path)
        if key in processed:
            continue

        rows.extend(process_file(path))
        processed.add(key)
        completed_this_run += 1

        if completed_this_run % 500 == 0:
            elapsed_min = (perf_counter() - t0) / 60
            overall_done = len(processed)
            rate = completed_this_run / max(elapsed_min, 1e-9)
            remaining = len(files) - overall_done
            eta_min = remaining / max(rate, 1e-9)
            print(
                f"{overall_done:,}/{len(files):,} files | "
                f"{len(rows):,} stays | "
                f"elapsed {elapsed_min:.1f} min | ETA ~{eta_min:.1f} min"
            )
            with PARTIAL_CACHE.open("wb") as f:
                pickle.dump(
                    {"processed_files": sorted(processed), "rows": rows},
                    f,
                    protocol=pickle.HIGHEST_PROTOCOL,
                )

    stays = pd.DataFrame(rows)
    stays["arrival_time_utc"] = pd.to_datetime(stays["arrival_time_utc"], utc=True)
    stays["departure_time_utc"] = pd.to_datetime(stays["departure_time_utc"], utc=True)
    stays = stays.sort_values(
        ["user_id", "arrival_time_utc", "source_file"], kind="stable"
    ).reset_index(drop=True)
    stays.to_pickle(STAYS_CACHE)
    if PARTIAL_CACHE.exists():
        PARTIAL_CACHE.unlink()
    print("Saved:", STAYS_CACHE)

print("Materialized stays:", f"{len(stays):,}")
print("Users with stays:", stays["user_id"].nunique())
assert len(stays) == EXPECTED_CP1_STAYS, (
    f"Expected {EXPECTED_CP1_STAYS} CP1 stays, got {len(stays)}"
)
display(stays.head())

## 2. User-level history sufficiency

Do not force Home/Office for every user.

Before semantic scoring, quantify how much repeated history each user actually has. All dates in this section are explicitly **UTC dates**, not behavioral local dates.

In [ ]:
user_history = (
    stays.assign(
        arrival_utc_date=stays["arrival_time_utc"].dt.date,
    )
    .groupby("user_id")
    .agg(
        stays=("user_id", "size"),
        active_utc_dates=("arrival_utc_date", "nunique"),
        first_stay_utc=("arrival_time_utc", "min"),
        last_stay_utc=("departure_time_utc", "max"),
        total_dwell_h=("duration_s", lambda s: s.sum() / 3600.0),
        median_stay_min=("duration_s", lambda s: s.median() / 60.0),
    )
)

user_history["observation_span_days"] = (
    user_history["last_stay_utc"] - user_history["first_stay_utc"]
).dt.total_seconds() / 86400.0

display(
    user_history[
        ["stays", "active_utc_dates", "observation_span_days", "total_dwell_h", "median_stay_min"]
    ].describe(percentiles=[.1,.25,.5,.75,.9,.95,.99])
)

print("Users with >=1 stay:", len(user_history))
for n in [2, 3, 5, 10]:
    print(f"Users with >= {n} stays:", int((user_history["stays"] >= n).sum()))
for n in [2, 3, 5, 10]:
    print(
        f"Users with stays on >= {n} distinct UTC dates:",
        int((user_history["active_utc_dates"] >= n).sum()),
    )

## 3. Candidate recurring-location representation

Home/Office is a property of a **recurring location across a user's history**, not of one `.plt` file.

This exploratory representation uses per-user Haversine DBSCAN:

- candidate radius: 200 m, matching the already reviewed recurrence scale;
- `min_samples=1` so every stay receives a location ID;
- recurrence is measured later by stay count / distinct dates.

This is **not frozen yet**. We inspect cluster spread and sensitivity before productionizing it.

DBSCAN can chain points, so a cluster may have members farther apart than the radius. We therefore report each cluster's max radius around its median representative.

In [ ]:
EARTH_RADIUS_M = 6_371_008.8
LOCATION_EPS_M = 200.0

def cluster_user_stays(group, eps_m=LOCATION_EPS_M):
    g = group.sort_values("arrival_time_utc", kind="stable").copy()
    coords_rad = np.radians(g[["latitude", "longitude"]].to_numpy(dtype=float))
    labels = DBSCAN(
        eps=eps_m / EARTH_RADIUS_M,
        min_samples=1,
        metric="haversine",
        algorithm="ball_tree",
    ).fit_predict(coords_rad)
    g["location_id"] = labels.astype(int)
    return g

cluster_parts = []
for user_id, group in stays.groupby("user_id", sort=True):
    clustered_user = cluster_user_stays(group)
    cluster_parts.append(clustered_user)

clustered = pd.concat(cluster_parts, ignore_index=True) if cluster_parts else stays.copy()

location_rows = []
for (user_id, location_id), g in clustered.groupby(["user_id", "location_id"], sort=True):
    lat = float(g["latitude"].median())
    lon = float(g["longitude"].median())
    radii = np.asarray(
        haversine_m(
            g["latitude"].to_numpy(dtype=float),
            g["longitude"].to_numpy(dtype=float),
            lat,
            lon,
        ),
        dtype=float,
    )
    location_rows.append({
        "user_id": user_id,
        "location_id": int(location_id),
        "latitude": lat,
        "longitude": lon,
        "stay_count": len(g),
        "active_utc_dates": g["arrival_time_utc"].dt.date.nunique(),
        "total_dwell_h": g["duration_s"].sum() / 3600.0,
        "max_radius_m": float(np.max(radii)) if len(radii) else 0.0,
    })

locations = pd.DataFrame(location_rows)

print("Users:", locations["user_id"].nunique())
print("Candidate locations:", len(locations))
print("Recurring locations (>=2 stays):", int((locations["stay_count"] >= 2).sum()))
print("Users with >=1 recurring location:", locations.loc[
    locations["stay_count"] >= 2, "user_id"
].nunique())

display(
    locations[
        ["stay_count", "active_utc_dates", "total_dwell_h", "max_radius_m"]
    ].describe(percentiles=[.5,.75,.9,.95,.99])
)

display(
    locations.sort_values("max_radius_m", ascending=False).head(20)
)

### Measured CP2 baseline findings — 2026-09-18

The first full-release materialization reproduced the frozen CP1 result exactly:

- **5,821 stays**
- **136 users with at least one stay**

User-history sufficiency:

- 120 users have >=2 stays;
- 99 users have >=5 stays;
- 81 users have >=10 stays;
- 114 users have stays on >=2 distinct UTC dates;
- 83 users have stays on >=5 distinct UTC dates;
- 62 users have stays on >=10 distinct UTC dates.

Candidate 200 m per-user DBSCAN representation produced:

- **1,885 candidate locations**;
- **635 recurring locations** with >=2 stays;
- **104 users** with at least one recurring location.

The largest cluster median-center radius reached about **526.7 m**, despite `eps=200 m`. This is expected DBSCAN chaining behavior and is a concrete reason not to freeze 200 m clustering semantics yet.

The first spatial summary also confirms that the stay table is strongly Beijing-centered but not Beijing-only. Stay coordinates include substantial geographic outliers, so a blanket UTC+8 conversion for the full release remains unjustified.

### Measured timezone/geography decision

Sensitivity on the 5,821 stays:

| radius | >=50% | >=80% | >=90% | >=95% |
| ---: | ---: | ---: | ---: | ---: |
| 50 km | 109 | 90 | 78 | 66 |
| 100 km | 113 | 97 | 87 | 78 |
| 200 km | 119 | 103 | 92 | 84 |

CP2 v1 freezes **100 km + 80% stay-share + 80% dwell-share** as an engineering cohort policy:

- 97 eligible users;
- 4,197 in-region stays;
- 245 travel/out-of-region stays excluded from otherwise eligible users;
- 72.10% of all materialized stays retained for semantic inference.

The policy is deliberately scoped and interpretable; it is not claimed to be an accuracy-optimal geographic boundary.

## 4. Timezone / geography gate — do not score Home/Office yet

GeoLife timestamps are UTC/GMT. Home and Office rules are behavioral-time rules.

A blanket:

```python
local_time = utc_time + 8 hours
```

is not acceptable for the full release because GeoLife contains trajectories outside Beijing.

This section only describes the spatial distribution of **stays**. It intentionally does not create local-hour features yet.

In [ ]:
spatial_summary = stays[["latitude", "longitude"]].describe(
    percentiles=[.01,.05,.25,.5,.75,.95,.99]
)
display(spatial_summary)

user_centers = (
    stays.groupby("user_id")[["latitude", "longitude"]]
    .median()
    .rename(columns={"latitude":"median_latitude","longitude":"median_longitude"})
)
display(user_centers.describe(percentiles=[.01,.05,.25,.5,.75,.95,.99]))

sample_n = min(5000, len(stays))
plot_sample = stays.sample(sample_n, random_state=42) if sample_n else stays
fig, ax = plt.subplots(figsize=(9, 6))
ax.scatter(plot_sample["longitude"], plot_sample["latitude"], s=8, alpha=0.35)
ax.set(
    title="Stay-point spatial coverage (sample; UTC semantics not yet converted)",
    xlabel="longitude",
    ylabel="latitude",
)
plt.show()

print("TIMEZONE POLICY STATUS: OPEN")
print("Do not run Home/Office time-of-day scoring until this gate is reviewed.")

### 4.1 Beijing-focused geography sensitivity

For a first interpretable Home/Office baseline, the safest option is to **scope semantic inference to a Beijing-focused cohort** rather than guess a timezone for every user.

This audit uses an approximate Beijing reference point only as a distance anchor:

```text
39.9042° N, 116.4074° E
```

It is **not** treated as an administrative boundary.

We measure three radii:

- 50 km;
- 100 km;
- 200 km.

For each user we compute:

- fraction of stays inside the radius;
- fraction of total dwell time inside the radius.

The candidate cohort rule is intentionally conservative and still **not frozen**:

```text
>= 80% of stays inside 100 km
AND
>= 80% of dwell time inside 100 km
```

Why both? A user may have many short Beijing stays but one very long stay elsewhere, or the reverse.

Users outside the candidate cohort are not assigned a timezone by guessing from longitude. They remain out-of-scope / abstain for the first Home/Office baseline.

In [ ]:
BEIJING_CENTER = (39.9042, 116.4074)
BEIJING_RADII_KM = [50.0, 100.0, 200.0]
CANDIDATE_RADIUS_KM = 100.0
CANDIDATE_MIN_STAY_SHARE = 0.80
CANDIDATE_MIN_DWELL_SHARE = 0.80

stays_geo = stays.copy()
stays_geo["distance_to_beijing_km"] = (
    np.asarray(
        haversine_m(
            stays_geo["latitude"].to_numpy(dtype=float),
            stays_geo["longitude"].to_numpy(dtype=float),
            BEIJING_CENTER[0],
            BEIJING_CENTER[1],
        ),
        dtype=float,
    )
    / 1000.0
)

print("Stay distance to Beijing reference point (km):")
display(
    stays_geo["distance_to_beijing_km"].describe(
        percentiles=[.5, .75, .9, .95, .99]
    )
)

radius_rows = []
user_geo_frames = {}

for radius_km in BEIJING_RADII_KM:
    inside = stays_geo["distance_to_beijing_km"] <= radius_km
    tmp = stays_geo.assign(
        inside_radius=inside,
        inside_dwell_s=np.where(inside, stays_geo["duration_s"], 0.0),
    )

    by_user = (
        tmp.groupby("user_id")
        .agg(
            total_stays=("user_id", "size"),
            inside_stays=("inside_radius", "sum"),
            total_dwell_s=("duration_s", "sum"),
            inside_dwell_s=("inside_dwell_s", "sum"),
        )
    )
    by_user["stay_share_inside"] = by_user["inside_stays"] / by_user["total_stays"]
    by_user["dwell_share_inside"] = (
        by_user["inside_dwell_s"] / by_user["total_dwell_s"]
    )
    user_geo_frames[radius_km] = by_user

    for min_share in [0.50, 0.80, 0.90, 0.95]:
        eligible = (
            (by_user["stay_share_inside"] >= min_share)
            & (by_user["dwell_share_inside"] >= min_share)
        )
        radius_rows.append({
            "radius_km": radius_km,
            "min_both_shares": min_share,
            "users": int(eligible.sum()),
            "stays_from_eligible_users": int(
                by_user.loc[eligible, "total_stays"].sum()
            ),
        })

radius_sensitivity = pd.DataFrame(radius_rows)
display(radius_sensitivity)

candidate_geo = user_geo_frames[CANDIDATE_RADIUS_KM].copy()
candidate_geo["beijing_candidate"] = (
    (candidate_geo["stay_share_inside"] >= CANDIDATE_MIN_STAY_SHARE)
    & (candidate_geo["dwell_share_inside"] >= CANDIDATE_MIN_DWELL_SHARE)
)

beijing_user_ids = candidate_geo.index[candidate_geo["beijing_candidate"]]
candidate_user_mask = stays_geo["user_id"].isin(beijing_user_ids)
inside_candidate_radius = (
    stays_geo["distance_to_beijing_km"] <= CANDIDATE_RADIUS_KM
)

# Only in-region stays receive Asia/Shanghai semantic-time treatment.
# Travel/out-of-region stays from otherwise Beijing-focused users remain excluded.
stays_beijing = stays_geo[candidate_user_mask & inside_candidate_radius].copy()
excluded_travel_stays = stays_geo[candidate_user_mask & ~inside_candidate_radius].copy()

print("Candidate Beijing users:", len(beijing_user_ids))
print("In-region candidate stays:", len(stays_beijing))
print("Excluded travel/out-of-region stays from candidate users:", len(excluded_travel_stays))
print(
    "Share of all materialized stays used for Beijing semantic audit:",
    f"{len(stays_beijing) / len(stays_geo):.2%}",
)

display(
    candidate_geo[
        [
            "total_stays",
            "inside_stays",
            "stay_share_inside",
            "dwell_share_inside",
            "beijing_candidate",
        ]
    ]
    .sort_values(
        ["beijing_candidate", "stay_share_inside", "dwell_share_inside"],
        ascending=[False, False, False],
    )
    .head(30)
)

### 4.2 Candidate local-time conversion for the Beijing cohort

Only after selecting the geography cohort do we convert UTC to local behavioral time.

For candidate Beijing users, only stays **inside the selected Beijing radius** enter the first semantic-time cohort. Out-of-radius travel stays remain excluded. For those in-region stays, the timezone is explicitly:

```text
Asia/Shanghai
```

Use timezone-aware conversion, not manual `+8h`.

This cell creates local timestamps for **audit only**. Home/Office scores are still not computed until the cohort sensitivity table is reviewed.

In [ ]:
BEIJING_TZ = ZoneInfo("Asia/Shanghai")

stays_beijing["arrival_time_local"] = (
    stays_beijing["arrival_time_utc"].dt.tz_convert(BEIJING_TZ)
)
stays_beijing["departure_time_local"] = (
    stays_beijing["departure_time_utc"].dt.tz_convert(BEIJING_TZ)
)
stays_beijing["arrival_local_date"] = stays_beijing["arrival_time_local"].dt.date
stays_beijing["arrival_local_hour"] = stays_beijing["arrival_time_local"].dt.hour
stays_beijing["arrival_local_weekday"] = (
    stays_beijing["arrival_time_local"].dt.weekday
)

print("Timezone:", BEIJING_TZ)
print("Users in candidate cohort:", stays_beijing["user_id"].nunique())
print("Stays in candidate cohort:", len(stays_beijing))

display(
    stays_beijing[
        [
            "user_id",
            "arrival_time_utc",
            "arrival_time_local",
            "departure_time_local",
            "duration_s",
        ]
    ].head(10)
)

display(
    stays_beijing["arrival_local_hour"]
    .value_counts()
    .sort_index()
    .rename("stays")
    .to_frame()
)

print("TIMEZONE POLICY STATUS: candidate Beijing cohort ready for review")
print("Home/Office scoring remains gated until geography sensitivity is reviewed.")

### 4.3 Recurring-location audit on the frozen Beijing semantic cohort

The earlier DBSCAN experiment was useful for discovering chaining, but it does not guarantee that all members of a 200 m cluster are within 200 m of one another.

For the semantic cohort, compare **complete-linkage agglomerative clustering** at 100 / 200 / 300 m.

With complete linkage and a distance threshold:

> every merge is constrained by the maximum pairwise distance between the two groups.

Therefore the final cluster diameter should not exceed the threshold (up to numerical tolerance).

This is closer to the semantic meaning we want for a recurring physical location than DBSCAN chaining.

The audit reports:

- number of candidate locations;
- recurring locations with >=2 stays;
- users with at least one recurring location;
- p95/max cluster diameter;
- sensitivity across 100/200/300 m.

No Home/Office score is computed in this section.

In [ ]:
COMPLETE_LINK_THRESHOLDS_M = [100.0, 200.0, 300.0]
CANDIDATE_COMPLETE_LINK_M = 200.0

def pairwise_haversine_matrix_m(group):
    lat = group["latitude"].to_numpy(dtype=float)
    lon = group["longitude"].to_numpy(dtype=float)
    return np.asarray(
        haversine_m(
            lat[:, None],
            lon[:, None],
            lat[None, :],
            lon[None, :],
        ),
        dtype=float,
    )

def complete_link_user(group, threshold_m):
    g = group.sort_values("arrival_time_local", kind="stable").copy()
    n = len(g)
    if n == 1:
        g["location_id"] = 0
        return g, np.zeros((1, 1), dtype=float)

    distances = pairwise_haversine_matrix_m(g)
    labels = AgglomerativeClustering(
        n_clusters=None,
        metric="precomputed",
        linkage="complete",
        distance_threshold=threshold_m,
    ).fit_predict(distances)
    g["location_id"] = labels.astype(int)
    return g, distances

def summarize_complete_link(threshold_m):
    clustered_parts = []
    location_rows = []

    for user_id, group in stays_beijing.groupby("user_id", sort=True):
        clustered_user, distances = complete_link_user(group, threshold_m)
        clustered_parts.append(clustered_user)

        labels = clustered_user["location_id"].to_numpy(dtype=int)
        for location_id in np.unique(labels):
            member_idx = np.flatnonzero(labels == location_id)
            members = clustered_user.iloc[member_idx]
            diameter_m = (
                float(distances[np.ix_(member_idx, member_idx)].max())
                if len(member_idx) > 1
                else 0.0
            )
            location_rows.append({
                "user_id": user_id,
                "location_id": int(location_id),
                "latitude": float(members["latitude"].median()),
                "longitude": float(members["longitude"].median()),
                "stay_count": len(members),
                "active_local_dates": members["arrival_local_date"].nunique(),
                "total_dwell_h": members["duration_s"].sum() / 3600.0,
                "diameter_m": diameter_m,
            })

    clustered_all = pd.concat(clustered_parts, ignore_index=True)
    locations_all = pd.DataFrame(location_rows)

    recurring = locations_all["stay_count"] >= 2
    return clustered_all, locations_all, {
        "threshold_m": threshold_m,
        "locations": len(locations_all),
        "recurring_locations": int(recurring.sum()),
        "users_with_recurring_location": int(
            locations_all.loc[recurring, "user_id"].nunique()
        ),
        "median_locations_per_user": float(
            locations_all.groupby("user_id").size().median()
        ),
        "p95_diameter_m": float(locations_all["diameter_m"].quantile(0.95)),
        "max_diameter_m": float(locations_all["diameter_m"].max()),
    }

cluster_sensitivity_rows = []
cluster_artifacts = {}

for threshold_m in COMPLETE_LINK_THRESHOLDS_M:
    clustered_threshold, locations_threshold, summary = summarize_complete_link(
        threshold_m
    )
    cluster_sensitivity_rows.append(summary)
    cluster_artifacts[threshold_m] = (
        clustered_threshold,
        locations_threshold,
    )

complete_link_sensitivity = pd.DataFrame(cluster_sensitivity_rows)
display(complete_link_sensitivity)

semantic_stays, semantic_locations = cluster_artifacts[CANDIDATE_COMPLETE_LINK_M]

assert semantic_locations["diameter_m"].max() <= CANDIDATE_COMPLETE_LINK_M + 1e-6

print("Candidate complete-link threshold:", CANDIDATE_COMPLETE_LINK_M, "m")
print("Semantic locations:", len(semantic_locations))
print(
    "Recurring semantic locations (>=2 stays):",
    int((semantic_locations["stay_count"] >= 2).sum()),
)
print(
    "Users with recurring semantic location:",
    semantic_locations.loc[
        semantic_locations["stay_count"] >= 2, "user_id"
    ].nunique(),
)
print(
    "Max verified cluster diameter (m):",
    semantic_locations["diameter_m"].max(),
)

display(
    semantic_locations.sort_values(
        ["stay_count", "total_dwell_h"],
        ascending=False,
    ).head(30)
)

## 5. Candidate heuristic parameters — declared, not frozen

After the timezone policy is reviewed, the first sensitivity grid can start from interpretable windows such as:

- Home evidence: late evening / overnight recurrence;
- Office evidence: weekday daytime recurrence, e.g. 09:00–17:00;
- minimum distinct days/nights;
- relevant-dwell share;
- margin over the second-ranked location.

These are parameters to test, not universal truths.

A user with weak history should produce **abstain / unknown**, not a forced label.

In [ ]:
CANDIDATE_HEURISTIC_CONFIG = {
    "timezone": "Asia/Shanghai",
    "beijing_reference_lat": BEIJING_CENTER[0],
    "beijing_reference_lon": BEIJING_CENTER[1],
    "beijing_radius_km": CANDIDATE_RADIUS_KM,
    "beijing_min_stay_share": CANDIDATE_MIN_STAY_SHARE,
    "beijing_min_dwell_share": CANDIDATE_MIN_DWELL_SHARE,
    "home_night_start_hour": 21,
    "home_night_end_hour": 6,
    "office_start_hour": 9,
    "office_end_hour": 17,
    "office_weekdays": [0, 1, 2, 3, 4],
    "candidate_location_complete_link_m": CANDIDATE_COMPLETE_LINK_M,
}

display(pd.Series(CANDIDATE_HEURISTIC_CONFIG, name="candidate_value"))
print("STATUS: candidate parameters only — do not freeze before timezone + recurrence review.")

## 6. CP2 review gate

Before implementing production Home/Office scoring:

1. `len(stays) == 5,821` reconciles with CP1 — **passed**;
2. user-history sufficiency — **measured**;
3. timezone/geography policy — **frozen for CP2 v1** at 100 km + 80/80%, in-region `Asia/Shanghai` only;
4. run/review complete-link recurring-location sensitivity at 100/200/300 m;
5. freeze recurring-location representation only if the 200 m engineering choice is reasonably stable and cluster diameter is verified;
6. define Home/Office scoring and abstention semantics;
7. write RED acceptance tests under `tests/`;
8. only then implement `src/geolife/model/`.

The notebook remains exploratory; production semantics belong in the contract and tests.